In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv
import os

In [35]:
load_dotenv()

True

In [ ]:
model = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)


In [42]:
# create a state

class LLMState(TypedDict):

    question: str
    answer: str

In [43]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [44]:
# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [45]:
# execute

intial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])

The average distance from the Earth to the Moon is about 384,400 kilometers (238,900 miles). This is called the 'lunar distance' or 'lunar mean distance.' However, the Moon's orbit is not a perfect circle and its distance from Earth varies slightly over the course of a month due to the elliptical shape of its orbit. At its closest point (called 'perigee'), the Moon is about 363,104 kilometers (225,623 miles) away, and at its farthest point (called 'apogee'), it is about 405,500 kilometers (252,088 miles) away.


In [46]:
model.invoke('How far is moon from the earth?').content

"The average distance from the Earth to the Moon is about 384,400 kilometers (238,900 miles). However, this distance can vary slightly due to the elliptical shape of the Moon's orbit around the Earth. At its closest point (called perigee), the Moon is about 363,104 kilometers (225,623 miles) away from the Earth, and at its farthest point (apogee), it is about 405,500 kilometers (252,088 miles) away."